# Cali Video Studio: Direct Audio-Driven Neural Lip-Sync Quickstart

Welcome to the **Cali Video Studio** neural lip-sync pipeline! This notebook allows you to generate high-fidelity, syllable-accurate talking head videos from **any single portrait image** and **any speech audio (`.wav`)** file on a free Google Colab GPU.

### Why this pipeline?
- **100% Syllable-Locked:** Mouth, jaw, and facial movement are computed directly from acoustic frequencies and speech phonemes.
- **Zero Credit Walls:** Completely free and open-source—no API costs or generation limits.
- **"Say Anything":** Works with any synthesized voice (Piper, Kokoro, ElevenLabs, etc.) or human recording.

---

### Step 1: Verify GPU Environment
Ensure your notebook is running on a GPU (`Runtime > Change runtime type > T4 GPU`).

In [ ]:
!nvidia-smi

### Step 2: Install Engine & Patches
This sets up SadTalker with compatibility patches for modern Python and Numpy runtimes.

In [ ]:
import os
if not os.path.exists('/content/SadTalker'):
    !git clone https://github.com/OpenTalker/SadTalker.git /content/SadTalker

%cd /content/SadTalker
!apt-get update -qq && apt-get install -y -qq ffmpeg libsndfile1

# Install core requirements without fragile packages
!pip install -q kornia face-alignment facexlib \
    imageio imageio-ffmpeg librosa numba resampy pydub yacs scikit-image safetensors av tqdm

# Patch GFPGAN import to avoid broken legacy C++ builds
!python -c "import pathlib; p = pathlib.Path('src/utils/face_enhancer.py'); t = p.read_text(); t = t.replace('from gfpgan import GFPGANer', 'try:\n    from gfpgan import GFPGANer\nexcept ImportError:\n    GFPGANer = None'); p.write_text(t); print('patched -- gfpgan import is now optional')"

# Compatibility patches for modern numpy
!python -c "import pathlib; p = pathlib.Path('src/face3d/util/preprocess.py'); t = p.read_text(); t = t.replace('np.VisibleDeprecationWarning', 'DeprecationWarning'); p.write_text(t); print('patched -- numpy deprecation')"
!python -c "import pathlib, re; p = pathlib.Path('src/face3d/util/my_awing_arch.py'); t = p.read_text(); t = re.sub(r'\bnp\.float\b', 'float', t); p.write_text(t); print('patched -- np.float')"
!python -c "import pathlib; p = pathlib.Path('src/face3d/util/preprocess.py'); t = p.read_text(); t = t.replace('trans_params = np.array([w0, h0, s, t[0], t[1]])', 'trans_params = np.array([float(np.asarray(v).reshape(-1)[0]) for v in [w0, h0, s, t[0], t[1]]])'); p.write_text(t); print('patched -- ragged array coercion')"

# Download pre-trained model weights
!bash scripts/download_models.sh

### Step 3: Upload Character Avatar & Speech Audio
Upload:
1. **Portrait Image:** A clear front-facing portrait (`.png` or `.jpg`).
2. **Speech Audio:** Spoken dialogue (`.wav`), ideally mastered to `-16 LUFS`.

In [ ]:
from google.colab import files
import shutil, os

upload_dir = '/content/SadTalker/inputs'
os.makedirs(upload_dir, exist_ok=True)
print('Upload your portrait image and speech audio (.wav):')
uploaded = files.upload()

image_name = None
audio_name = None

for name in uploaded:
    dest = os.path.join(upload_dir, name)
    shutil.move(name, dest)
    if name.lower().endswith(('.png', '.jpg', '.jpeg', '.webp')):
        image_name = name
    elif name.lower().endswith(('.wav', '.mp3', '.aac')):
        audio_name = name

print(f'Ready! Avatar: {image_name} | Speech Audio: {audio_name}')

### Step 4: Generate Syllable-Locked Talking Video
Run inference to produce the lip-synced video.

In [ ]:
%cd /content/SadTalker
import time, glob
from IPython.display import Video, display
from google.colab import files as colab_files

assert image_name and audio_name, 'Please upload both an image and an audio file!'

output_dir = 'results/quickstart'
os.makedirs(output_dir, exist_ok=True)

print('Starting neural lip-sync generation...')
start_time = time.time()

!python inference.py \
    --driven_audio inputs/{audio_name} \
    --source_image inputs/{image_name} \
    --result_dir {output_dir} \
    --still \
    --preprocess full \
    --size 256

elapsed = time.time() - start_time
print(f'Generation completed in {elapsed/60:.1f} minutes!')

# Locate and display output video
output_videos = sorted(glob.glob(f'{output_dir}/**/*.mp4', recursive=True))
if output_videos:
    latest_video = output_videos[-1]
    print(f'Generated Video: {latest_video}')
    display(Video(latest_video, embed=True, width=420))
    colab_files.download(latest_video)
else:
    print('Error: Video generation failed. Check inference logs above.')